In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import os

path = kagglehub.dataset_download("arashnic/hr-ana")
df = pd.read_csv(os.path.join(path, "train.csv"))

df = df[[
    "department",
    "education",
    "gender",
    "recruitment_channel",
    "no_of_trainings",
    "age",
    "previous_year_rating",
    "length_of_service",
    "awards_won?",
    "avg_training_score",
    "is_promoted"
]].copy()

df["no_of_trainings"] = pd.cut(
    df["no_of_trainings"],
    bins=[0, 2, 4, 10],
    labels=["Low", "Medium", "High"]
)

df["age"] = pd.cut(
    df["age"],
    bins=[0, 30, 40, 100],
    labels=["Young", "Middle", "Senior"]
)

df["previous_year_rating"] = df["previous_year_rating"].map({
    1: "Low",
    2: "Low",
    3: "Medium",
    4: "High",
    5: "High"
})

df["length_of_service"] = pd.cut(
    df["length_of_service"],
    bins=[0, 5, 10, 40],
    labels=["Short", "Medium", "Long"]
)

df["awards_won?"] = df["awards_won?"].map({
    0: "No",
    1: "Yes"
})

df["avg_training_score"] = pd.cut(
    df["avg_training_score"],
    bins=[0, 50, 70, 100],
    labels=["Low", "Medium", "High"]
)

df["is_promoted"] = df["is_promoted"].map({
    0: "No",
    1: "Yes"
})

df = df.dropna()
df = df.head(1000).reset_index(drop=True)

attributes = list(df.columns[:-1])
target = df.columns[-1]

ANY = "?"
NULL = "Ø"

def covers(h, x):
    for hv, xv in zip(h, x):
        if hv == NULL:
            return False
        if hv == ANY:
            continue
        if hv != xv:
            return False
    return True

def more_general_or_equal(h1, h2):
    for a, b in zip(h1, h2):
        if a == ANY:
            continue
        if a == NULL:
            if b != NULL:
                return False
        else:
            if b == ANY:
                return False
            if b == NULL:
                continue
            if a != b:
                return False
    return True

domains = {
    attr: list(df[attr].unique())
    for attr in attributes
}

def generalize_S(s, x):
    s = list(s)

    for i in range(len(s)):
        if s[i] == NULL:
            s[i] = x[i]
        elif s[i] != x[i]:
            s[i] = ANY

    return tuple(s)

def specialize_G(g, x, s):
    specializations = []

    for i, attr in enumerate(attributes):
        if g[i] == ANY:
            if s[i] != ANY and s[i] != NULL:
                for value in domains[attr]:
                    if value != x[i]:
                        new_h = list(g)
                        new_h[i] = value
                        specializations.append(tuple(new_h))

    return specializations

def remove_more_general_from_S(S):
    result = []

    for s in S:
        is_more_general = False

        for s2 in S:
            if s != s2 and more_general_or_equal(s, s2):
                is_more_general = True
                break

        if not is_more_general:
            result.append(s)

    return result

def remove_more_specific_from_G(G):
    result = []

    for g in G:
        is_more_specific = False

        for g2 in G:
            if g != g2 and more_general_or_equal(g2, g):
                is_more_specific = True
                break

        if not is_more_specific:
            result.append(g)

    return result

S = [tuple([NULL] * len(attributes))]
G = [tuple([ANY] * len(attributes))]

for index, row in df.iterrows():

    x = tuple(row[attr] for attr in attributes)
    label = row[target]

    if label == "Yes":

        G = [
            g for g in G
            if covers(g, x)
        ]

        new_S = []

        for s in S:

            if covers(s, x):
                new_S.append(s)

            else:
                generalized_s = generalize_S(s, x)

                for g in G:
                    if more_general_or_equal(g, generalized_s):
                        new_S.append(generalized_s)
                        break

        S = list(set(new_S))
        S = remove_more_general_from_S(S)

    elif label == "No":

        S = [
            s for s in S
            if not covers(s, x)
        ]

        new_G = []

        for g in G:

            if not covers(g, x):
                new_G.append(g)

            else:

                specializations = specialize_G(
                    g,
                    x,
                    S[0] if len(S) > 0 else tuple(
                        [NULL] * len(attributes)
                    )
                )

                for new_g in specializations:
                    if any(
                        more_general_or_equal(new_g, s)
                        for s in S
                    ):
                        new_G.append(new_g)

        G = list(set(new_G))
        G = remove_more_specific_from_G(G)

print("\nFinal Specific Boundary (S):")

if len(S) == 0:
    print("S = EMPTY")
else:
    for h in S:
        print(h)

print("\nFinal General Boundary (G):")

if len(G) == 0:
    print("G = EMPTY")
else:
    for h in G:
        print(h)

print("\nNumber of Training Examples:", len(df))
print("Number of Promoted Employees:", (df[target] == "Yes").sum())
print("Number of Non-Promoted Employees:", (df[target] == "No").sum())

y = df[target].map({
    "Yes": 1,
    "No": 0
})

plt.figure(figsize=(12, 5))

plt.plot(
    range(1, len(df) + 1),
    y,
    marker="o",
    markersize=2,
    linewidth=1
)

plt.yticks(
    [0, 1],
    ["Not Promoted", "Promoted"]
)

plt.xticks(
    range(0, len(df) + 1, 100)
)

plt.xlabel("Employee Number")
plt.ylabel("Promotion Status")
plt.title("Candidate-Elimination: Promotion and Non-Promotion Examples")
plt.grid(True)

plt.show()

Using Colab cache for faster access to the 'hr-ana' dataset.


In [ ]:
is this